In [1]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [2]:
import subprocess
subprocess.Popen("ollama serve", stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, shell=True)
import time
time.sleep(3) # Wait for a few seconds for Ollama to load!

In [3]:
!ollama pull llama3.1:8b

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest ⠏ pulling manifest ⠋ pulling manifest 
pulling 667b0c1932bc:   0% ▕                  ▏ 8.7 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   1% ▕                  ▏  46 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   2% ▕                  ▏ 120 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   4% ▕                  ▏ 198 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   6% ▕█                 ▏ 284 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   7% ▕█                 ▏ 330 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   8% ▕█                 ▏ 411 MB/4.9 GB                  pulling manifest 
pulling 667b0c1932bc:   9% ▕█                 ▏ 462 MB/4.9 GB                  pulling manifes

In [4]:
%%capture
!pip install dspy
! pip install litellm[proxy]

In [5]:
import dspy
lm = dspy.LM('ollama_chat/llama3.1:8b', api_base='http://localhost:11434', api_key='')
dspy.configure(lm=lm)

In [6]:
import logging

for name in logging.root.manager.loggerDict:
    if 'litellm' in name.lower() or 'ollama' in name.lower():
        # print(name)
        logging.getLogger(name).setLevel(logging.ERROR)

for name in logging.root.manager.loggerDict:
    if 'dspy' in name.lower():
        # print(name)
        logging.getLogger(name).setLevel(logging.INFO)

In [7]:
from typing import Literal, List
import dspy

class ClassifyDiabetesStigma(dspy.Signature):
    """
    Given a social media post (title and body), classify whether it includes stigma toward people with diabetes.
    """

    title: str = dspy.InputField(desc="The title of the social media post.")
    post: str = dspy.InputField(desc="The main text or content of the social media post.")

    category: Literal['yes stigma', 'no stigma'] = dspy.OutputField(
        desc="Does the post contain diabetes-related stigma?"
    )

    confidence: float = dspy.OutputField(desc="Model's confidence score.")




class ClassifyDiabetesStigmaType(dspy.Signature):
    """
    Given a social media post (title and body) that is already detected to contain stigma toward people with diabetes,
    identify the types of stigma present.
    """

    title: str = dspy.InputField(desc="The title of the social media post.")
    post: str = dspy.InputField(desc="The main text or content of the social media post.")

    stigma_type: List[Literal[
        'experienced stigma',
        'perceived stigma',
        'anticipated stigma',
        'internalized stigma',
        'intersectional stigma'
    ]] = dspy.OutputField(
        desc="List of stigma types present in the post."
    )

    confidence: float = dspy.OutputField(desc="Model's confidence score.")



In [8]:
classify_stigma = dspy.Predict(ClassifyDiabetesStigma)
classify_stigma(title = 'I don\'t know what t do', post='When I first got diagnosed at 16, I craved validation for the hurt I was feeling. There was none. I just saw article after article saying I could “Live a normal life”. I would write letters to myself about how angry I was and feel like an idiot. I started drinking by myself. I wish this had been here when I was 16, I’m so grateful it is for the next generation.')

Prediction(
    category='yes stigma',
    confidence=0.8
)

In [10]:
import json
with open('/kaggle/working/train_data.json', 'r') as f:
  train_data = json.loads(f.read())
  f.close()

with open('/kaggle/working/val_data.json', 'r') as f:
  val_data = json.loads(f.read())
  f.close()

In [11]:
f_train_data = [d for d in train_data if d['label'] == 'yes stigma']
f_val_data = [d for d in val_data if d['label'] == 'yes stigma']

In [12]:
l_trainset = []

for post_obj in f_train_data:
    example = dspy.Example(title = post_obj['title'], post=post_obj['post text'], category=post_obj['label']).with_inputs("title", "post")
    l_trainset.append(example)


l_valset = []

for post_obj in f_val_data:
    example = dspy.Example(title = post_obj['title'], post=post_obj['post text'], category=post_obj['label']).with_inputs("title", "post")
    l_valset.append(example)

In [13]:
prompt_gen_lm = dspy.LM('ollama_chat/llama3.1:8b', api_base='http://localhost:11434', api_key='')

In [14]:
from sklearn.metrics import f1_score
import numpy as np

def accuracy_metric(example, prediction, trace=None):
    return prediction.category == example.category

In [15]:
def combined_f1_metric(example, prediction, trace=None):
    # ----- CATEGORY (single-label classification) -----
    true_cat = example['category']
    pred_cat = prediction.category

    category_f1 = f1_score([true_cat], [pred_cat], average='macro', zero_division=1)

    # ----- STIGMA TYPE (multi-label classification) -----
    true_stigma = example.get('stigma_type', [])
    pred_stigma = prediction.stigma_type if prediction.stigma_type is not None else []

    # Align possible labels
    all_labels = ['experienced stigma', 'perceived stigma', 'anticipated stigma', 'internalized stigma', 'intersectional stigma']

    # Convert to binary format for multi-label F1
    def to_multilabel(y, labels):
        return [1 if label in y else 0 for label in labels]

    y_true_bin = [to_multilabel(true_stigma, all_labels)]
    y_pred_bin = [to_multilabel(pred_stigma, all_labels)]

    stigma_f1 = f1_score(y_true_bin, y_pred_bin, average='micro', zero_division=1)

    # ----- COMBINED SCORE -----
    combined_score = 0.5 * category_f1 + 0.5 * stigma_f1
    return combined_score


# Label detection optimization

# Bootstap few-shot

In [16]:
from dspy.teleprompt import BootstrapFewShot

fewshot_optimizer = BootstrapFewShot(metric=accuracy_metric, max_bootstrapped_demos=8, max_labeled_demos=16)

optimized_classify = fewshot_optimizer.compile(student = classify_stigma, trainset=l_trainset)

 18%|█▊        | 18/100 [00:59<04:32,  3.32s/it]

Bootstrapped 8 full traces after 18 examples for up to 1 rounds, amounting to 18 attempts.


In [17]:
optimized_classify.save('Llama_label_bootstrap_fewshot_optimized.json')

In [18]:
del optimized_classify

# Bootstrap fewshot random search

In [19]:
from dspy.teleprompt import BootstrapFewShotWithRandomSearch

fewshot_optimizer = BootstrapFewShotWithRandomSearch(metric=accuracy_metric, max_bootstrapped_demos=8, max_labeled_demos=16, num_candidate_programs=3, num_threads=3)

optimized_classify = fewshot_optimizer.compile(student = classify_stigma, trainset=l_trainset, valset=l_valset)

Going to sample between 1 and 8 traces per predictor.
Will attempt to bootstrap 3 candidate sets.
Average Metric: 8.00 / 10 (80.0%): 100%|██████████| 10/10 [00:13<00:00,  1.32s/it]

2025/08/10 15:49:31 INFO dspy.evaluate.evaluate: Average Metric: 8 / 10 (80.0%)



New best score: 80.0 for seed -3
Scores so far: [80.0]
Best score so far: 80.0
Average Metric: 8.00 / 10 (80.0%): 100%|██████████| 10/10 [00:41<00:00,  4.11s/it]

2025/08/10 15:50:12 INFO dspy.evaluate.evaluate: Average Metric: 8 / 10 (80.0%)



Scores so far: [80.0, 80.0]
Best score so far: 80.0


...


Scores so far: [80.0, 80.0, 100.0, 80.0, 60.0]
Best score so far: 100.0


  4%|▍         | 4/100 [00:11<04:36,  2.88s/it]


Bootstrapped 1 full traces after 4 examples for up to 1 rounds, amounting to 4 attempts.
Average Metric: 8.00 / 10 (80.0%): 100%|██████████| 10/10 [00:37<00:00,  3.79s/it]

2025/08/10 15:54:43 INFO dspy.evaluate.evaluate: Average Metric: 8 / 10 (80.0%)



Scores so far: [80.0, 80.0, 100.0, 80.0, 60.0, 80.0]
Best score so far: 100.0
6 candidate programs found.


In [20]:
optimized_classify.save('Llama_label_bootstrap_fewshot_random_search_optimized.json')

In [21]:
del optimized_classify

# MIPROV2

In [22]:
tp = dspy.MIPROv2(metric=accuracy_metric,
                  auto = None,
                  prompt_model=prompt_gen_lm,
                  num_candidates=3,
                  init_temperature=0,
                  task_model=lm)

optimized_classify = tp.compile(classify_stigma, trainset=l_trainset, valset=l_valset,
                  max_labeled_demos=16,
                  max_bootstrapped_demos=8,
                  minibatch_size=10,
                  num_trials=6,
                  minibatch_full_eval_steps=10,
                  requires_permission_to_run=False)

2025/08/10 15:54:43 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2025/08/10 15:54:43 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2025/08/10 15:54:43 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=3 sets of demonstrations...


Bootstrapping set 1/3
Bootstrapping set 2/3
Bootstrapping set 3/3


 18%|█▊        | 18/100 [00:00<00:00, 241.78it/s]
2025/08/10 15:54:43 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 2: PROPOSE INSTRUCTION CANDIDATES <==
2025/08/10 15:54:43 INFO dspy.teleprompt.mipro_optimizer_v2: We will use the few-shot examples from the previous step, a generated dataset summary, a summary of the program code, and a randomly selected prompting tip to propose instructions.


Bootstrapped 8 full traces after 18 examples for up to 1 rounds, amounting to 18 attempts.
Error getting source code: unhashable type: 'dict'.

Running without program aware proposer.


2025/08/10 15:56:40 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.
2025/08/10 15:58:34 INFO dspy.teleprompt.mipro_optimizer_v2: 
Proposing N=3 instructions...

2025/08/10 15:58:51 INFO dspy.teleprompt.mipro_optimizer_v2: Proposed Instructions for Predictor 0:

2025/08/10 15:58:51 INFO dspy.teleprompt.mipro_optimizer_v2: 0: Given a social media post (title and body), classify whether it includes stigma toward people with diabetes.

2025/08/10 15:58:51 INFO dspy.teleprompt.mipro_optimizer_v2: 1: Classify a social media post as including stigma toward people with diabetes or not, based on its title and body.

The instruction should be clear and concise, focusing on identifying stigmatizing language or sentiments towards individuals living with diabetes in the given text.

2025/08/10 15:58:51 INFO dspy.teleprompt.mipro_optimizer_v2: 2: Identify posts that express stigma or shame related to living with type 2 diabetes. You are a social

...

2025/08/10 16:00:36 INFO dspy.evaluate.evaluate: Average Metric: 10 / 10 (100.0%)
2025/08/10 16:00:36 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 100.0 on minibatch of size 10 with parameters ['Predictor 0: Instruction 2', 'Predictor 0: Few-Shot Set 2'].
2025/08/10 16:00:36 INFO dspy.teleprompt.mipro_optimizer_v2: Minibatch scores so far: []
2025/08/10 16:00:36 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [80.0, 90.0, 100.0, 80.0, 70.0, 100.0]
2025/08/10 16:00:36 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 80.0
2025/08/10 16:00:36 INFO dspy.teleprompt.mipro_optimizer_v2: ========================================


2025/08/10 16:00:36 INFO dspy.teleprompt.mipro_optimizer_v2: == Trial 7 / 8 - Minibatch ==



Average Metric: 10.00 / 10 (100.0%): 100%|██████████| 10/10 [00:00<00:00, 1240.11it/s]

2025/08/10 16:00:36 INFO dspy.evaluate.evaluate: Average Metric: 10 / 10 (100.0%)
2025/08/10 16:00:36 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 100.0 on minibatch of size 10 with parameters ['Predictor 0: Instruction 2', 'Predictor 0: Few-Shot Set 2'].
2025/08/10 16:00:36 INFO dspy.teleprompt.mipro_optimizer_v2: Minibatch scores so far: []
2025/08/10 16:00:36 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [80.0, 90.0, 100.0, 80.0, 70.0, 100.0, 100.0]
2025/08/10 16:00:36 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 80.0
2025/08/10 16:00:36 INFO dspy.teleprompt.mipro_optimizer_v2: ========================================


2025/08/10 16:00:36 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Trial 8 / 8 - Full Evaluation =====
2025/08/10 16:00:36 INFO dspy.teleprompt.mipro_optimizer_v2: Doing full eval on next top averaging program (Avg Score: 100.0) from minibatch trials...



Average Metric: 10.00 / 10 (100.0%): 100%|██████████| 10/10 [00:00<00:00, 771.74it/s]

2025/08/10 16:00:36 INFO dspy.evaluate.evaluate: Average Metric: 10 / 10 (100.0%)
2025/08/10 16:00:36 INFO dspy.teleprompt.mipro_optimizer_v2: New best full eval score! Score: 100.0
2025/08/10 16:00:36 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [80.0, 90.0, 100.0, 80.0, 70.0, 100.0, 100.0, 100.0]
2025/08/10 16:00:36 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 100.0
2025/08/10 16:00:36 INFO dspy.teleprompt.mipro_optimizer_v2: =======================
2025/08/10 16:00:36 INFO dspy.teleprompt.mipro_optimizer_v2: 

2025/08/10 16:00:36 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 100.0!


In [23]:
optimized_classify.save('Llama_label_MIPROV2_optimized.json')

In [24]:
del optimized_classify

# COPRO

In [25]:
import litellm
litellm.drop_params = True

In [26]:
import litellm
litellm.drop_params = True

from dspy.teleprompt import COPRO

eval_kwargs = dict(num_threads=16, display_progress=True, display_table=0)

copro_teleprompter = COPRO(prompt_model=prompt_gen_lm, metric=accuracy_metric, max_bootstrapped_demos=8, max_labeled_demos=16, breadth=5, depth=3, init_temperature=0, verbose=False)

optimized_classify = copro_teleprompter.compile(classify_stigma, trainset=l_trainset, eval_kwargs=eval_kwargs)

2025/08/10 16:00:43 INFO dspy.teleprompt.copro_optimizer: Iteration Depth: 1/3.
2025/08/10 16:00:43 INFO dspy.teleprompt.copro_optimizer: At Depth 1/3, Evaluating Prompt Candidate #1/2 for Predictor 1 of 1.






[2025-08-10T16:00:43.219748]

System message:

Your input fields are:
1. `basic_instruction` (str): The initial instructions before optimization
Your output fields are:
1. `proposed_instruction` (str): The improved instructions for the language model
2. `proposed_prefix_for_output_field` (str): The string at the end of the prompt, which will help the model start solving the task
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## basic_instruction ## ]]
{basic_instruction}

[[ ## proposed_instruction ## ]]
{proposed_instruction}

[[ ## proposed_prefix_for_output_field ## ]]
{proposed_prefix_for_output_field}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        You are an instruction optimizer for large language models. I will give you a ``signature`` of fields (inputs and outputs) in English. Your task is to propose an instruction that will lead a good language model to perform the task well. Don't be 

2025/08/10 16:00:50 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


Average Metric: 49.00 / 100 (49.0%): 100%|██████████| 100/100 [02:19<00:00,  1.39s/it]

2025/08/10 16:03:02 INFO dspy.evaluate.evaluate: Average Metric: 49 / 100 (49.0%)
2025/08/10 16:03:02 INFO dspy.teleprompt.copro_optimizer: At Depth 1/3, Evaluating Prompt Candidate #2/2 for Predictor 1 of 1.


...






[2025-08-10T16:05:22.617235]

System message:

Your input fields are:
1. `attempted_instructions` (str):
Your output fields are:
1. `proposed_instruction` (str): The improved instructions for the language model
2. `proposed_prefix_for_output_field` (str): The string at the end of the prompt, which will help the model start solving the task
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## attempted_instructions ## ]]
{attempted_instructions}

[[ ## proposed_instruction ## ]]
{proposed_instruction}

[[ ## proposed_prefix_for_output_field ## ]]
{proposed_prefix_for_output_field}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        You are an instruction optimizer for large language models. I will give some task instructions I've tried, along with their corresponding validation scores. The instructions are arranged in increasing order based on their scores, where higher scores indicate better quality.


2025/08/10 16:09:35 INFO dspy.teleprompt.copro_optimizer: Iteration Depth: 3/3.
2025/08/10 16:09:35 INFO dspy.teleprompt.copro_optimizer: At Depth 3/3, Evaluating Prompt Candidate #1/1 for Predictor 1 of 1.


Average Metric: 64.00 / 100 (64.0%): 100%|██████████| 100/100 [05:29<00:00,  3.29s/it]

2025/08/10 16:15:04 INFO dspy.evaluate.evaluate: Average Metric: 64 / 100 (64.0%)







[2025-08-10T16:09:35.028119]

System message:

Your input fields are:
1. `attempted_instructions` (str):
Your output fields are:
1. `proposed_instruction` (str): The improved instructions for the language model
2. `proposed_prefix_for_output_field` (str): The string at the end of the prompt, which will help the model start solving the task
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## attempted_instructions ## ]]
{attempted_instructions}

[[ ## proposed_instruction ## ]]
{proposed_instruction}

[[ ## proposed_prefix_for_output_field ## ]]
{proposed_prefix_for_output_field}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        You are an instruction optimizer for large language models. I will give some task instructions I've tried, along with their corresponding validation scores. The instructions are arranged in increasing order based on their scores, where higher scores indicate better quality.


In [27]:
optimized_classify.save('Llama_label_COPRO_optimized.json')

In [28]:
del optimized_classify

# Bootstrap few-shot with optuna

In [29]:
from dspy.teleprompt import BootstrapFewShotWithOptuna

fewshot_optuna_optimizer = BootstrapFewShotWithOptuna(metric=accuracy_metric, max_bootstrapped_demos=8, max_labeled_demos=16, num_candidate_programs=3)

optimized_classify = fewshot_optuna_optimizer.compile(student=classify_stigma, trainset=l_trainset, valset=l_valset, max_demos = 8)

Going to sample between 1 and 8 traces per predictor.
Will attempt to train 3 candidate sets.


 18%|█▊        | 18/100 [00:00<00:00, 239.38it/s]


Bootstrapped 8 full traces after 18 examples for up to 1 rounds, amounting to 18 attempts.
Average Metric: 6.00 / 10 (60.0%): 100%|██████████| 10/10 [00:15<00:00,  1.59s/it]

2025/08/10 16:15:20 INFO dspy.evaluate.evaluate: Average Metric: 6 / 10 (60.0%)



Average Metric: 2.00 / 3 (66.7%):  30%|███       | 3/10 [00:06<00:12,  1.85s/it]

...

Average Metric: 3.00 / 4 (75.0%):  40%|████      | 4/10 [00:11<00:20,  3.44s/it]

2025/08/10 16:15:54 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


Average Metric: 7.00 / 10 (70.0%): 100%|██████████| 10/10 [00:25<00:00,  2.50s/it]

2025/08/10 16:16:05 INFO dspy.evaluate.evaluate: Average Metric: 7 / 10 (70.0%)



Best score: 70.0
Best program: Predict(ClassifyDiabetesStigma(title, post -> category, confidence
    instructions='Given a social media post (title and body), classify whether it includes stigma toward people with diabetes.'
    title = Field(annotation=str required=True json_schema_extra={'desc': 'The title of the social media post.', '__dspy_field_type': 'input', 'prefix': 'Title:'})
    post = Field(annotation=str required=True json_schema_extra={'desc': 'The main text or content of the social media post.', '__dspy_field_type': 'input', 'prefix': 'Post:'})
    category = Field(annotation=Literal['yes stigma', 'no stigma'] required=True json_schema_extra={'desc': 'Does the post contain diabetes-related stigma?', '__dspy_field_type': 'output', 'prefix': 'Category:'})
    confidence = Field(annotation=float required=True json_schema_extra={'desc': "Model's confidence score.", '__dspy_field_type': 'output', 'prefix': 'Confidence:'})
))


In [30]:
optimized_classify.save('Llama_label_bootstrap_fewshot_optuna_optimized.json')

In [31]:
del optimized_classify

# Labled KNN fewshot

In [32]:
from dspy.teleprompt import LabeledFewShot

labeled_fewshot_optimizer = LabeledFewShot(k=8)
optimized_classify = labeled_fewshot_optimizer.compile(student = classify_stigma, trainset=l_trainset)

In [33]:
optimized_classify.save('Llama_label_labeled_fewshot_optimized.json')

In [34]:
del optimized_classify

# Type detection optimization

In [35]:
class ClassifyDiabetesStigmaType(dspy.Signature):
    """
    Given a social media post (title and body) that is already detected to contain stigma toward people with diabetes,
    identify the types of stigma present.
    """

    title: str = dspy.InputField(desc="The title of the social media post.")
    post: str = dspy.InputField(desc="The main text or content of the social media post.")

    stigma_type: List[Literal[
        'experienced stigma',
        'perceived stigma',
        'anticipated stigma',
        'internalized stigma',
        'intersectional stigma'
    ]] = dspy.OutputField(
        desc="List of stigma types present in the post."
    )

    confidence: float = dspy.OutputField(desc="Model's confidence score.")



In [36]:
type_detector = dspy.Predict(ClassifyDiabetesStigmaType)
type_detector(title = 'I don\'t know what t do', post='When I first got diagnosed at 16, I craved validation for the hurt I was feeling. There was none. I just saw article after article saying I could “Live a normal life”. I would write letters to myself about how angry I was and feel like an idiot. I started drinking by myself. I wish this had been here when I was 16, I’m so grateful it is for the next generation.')

Prediction(
    stigma_type=['experienced stigma', 'perceived stigma'],
    confidence=0.8
)

In [37]:
t_trainset = []

for post_obj in f_train_data:
    example = dspy.Example(title = post_obj['title'], post=post_obj['post text'], stigma_type=post_obj['types']).with_inputs("title", "post")
    t_trainset.append(example)


t_valset = []

for post_obj in f_val_data:
    example = dspy.Example(title = post_obj['title'], post=post_obj['post text'], stigma_type=post_obj['types']).with_inputs("title", "post")
    t_valset.append(example)

In [38]:
def jaccard_metric(example, prediction, trace=None):
    set_p = set(prediction.stigma_type)
    set_e = set(example.stigma_type)
    return len(set_p.intersection(set_e)) / len(set_p.union(set_e))

# Bootstrap fewshot

In [39]:
from dspy.teleprompt import BootstrapFewShot

fewshot_optimizer = BootstrapFewShot(metric=jaccard_metric, max_bootstrapped_demos=8, max_labeled_demos=16)

optimized_classify = fewshot_optimizer.compile(student = type_detector, trainset=t_trainset)

 13%|█▎        | 13/100 [00:57<06:27,  4.46s/it]

Bootstrapped 8 full traces after 13 examples for up to 1 rounds, amounting to 13 attempts.


In [40]:
optimized_classify.save('Llama_type_bootstrap_fewshot_optimized.json')

In [41]:
del optimized_classify

# Bootstrap fewshot random search

In [42]:
from dspy.teleprompt import BootstrapFewShotWithRandomSearch

fewshot_optimizer = BootstrapFewShotWithRandomSearch(metric=jaccard_metric, max_bootstrapped_demos=8, max_labeled_demos=16, num_candidate_programs=3, num_threads=3)

optimized_classify = fewshot_optimizer.compile(student = type_detector, trainset=t_trainset, valset=t_valset)

Going to sample between 1 and 8 traces per predictor.
Will attempt to bootstrap 3 candidate sets.
Average Metric: 2.50 / 10 (25.0%): 100%|██████████| 10/10 [00:17<00:00,  1.74s/it]

2025/08/10 16:17:26 INFO dspy.evaluate.evaluate: Average Metric: 2.5 / 10 (25.0%)



New best score: 25.0 for seed -3
Scores so far: [25.0]
Best score so far: 25.0
Average Metric: 3.25 / 10 (32.5%): 100%|██████████| 10/10 [00:48<00:00,  4.85s/it]

2025/08/10 16:18:15 INFO dspy.evaluate.evaluate: Average Metric: 3.25 / 10 (32.5%)



New best score: 32.5 for seed -2
Scores so far: [25.0, 32.5]
Best score so far: 32.5


...


Scores so far: [25.0, 32.5, 20.83, 32.5, 32.5]
Best score so far: 32.5


  3%|▎         | 3/100 [00:21<11:38,  7.20s/it]


Bootstrapped 1 full traces after 3 examples for up to 1 rounds, amounting to 3 attempts.
Average Metric: 4.25 / 10 (42.5%): 100%|██████████| 10/10 [01:08<00:00,  6.84s/it]

2025/08/10 16:23:15 INFO dspy.evaluate.evaluate: Average Metric: 4.25 / 10 (42.5%)



New best score: 42.5 for seed 2
Scores so far: [25.0, 32.5, 20.83, 32.5, 32.5, 42.5]
Best score so far: 42.5
6 candidate programs found.


In [43]:
optimized_classify.save('Llama_type_bootstrap_fewshot_random_search_optimized.json')

In [44]:
del optimized_classify

# MIPROV2

In [45]:
tp = dspy.MIPROv2(metric=jaccard_metric,
                  auto = None,
                  prompt_model=prompt_gen_lm,
                  num_candidates=3,
                  init_temperature=0,
                  task_model=lm)

optimized_classify = tp.compile(type_detector, trainset=t_trainset, valset=t_valset,
                  max_labeled_demos=16,
                  max_bootstrapped_demos=8,
                  minibatch_size=10,
                  num_trials=6,
                  minibatch_full_eval_steps=10,
                  requires_permission_to_run=False)

2025/08/10 16:23:16 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 1: BOOTSTRAP FEWSHOT EXAMPLES <==
2025/08/10 16:23:16 INFO dspy.teleprompt.mipro_optimizer_v2: These will be used as few-shot example candidates for our program and for creating instructions.

2025/08/10 16:23:16 INFO dspy.teleprompt.mipro_optimizer_v2: Bootstrapping N=3 sets of demonstrations...


Bootstrapping set 1/3
Bootstrapping set 2/3
Bootstrapping set 3/3


 13%|█▎        | 13/100 [00:00<00:00, 179.30it/s]
2025/08/10 16:23:16 INFO dspy.teleprompt.mipro_optimizer_v2: 
==> STEP 2: PROPOSE INSTRUCTION CANDIDATES <==
2025/08/10 16:23:16 INFO dspy.teleprompt.mipro_optimizer_v2: We will use the few-shot examples from the previous step, a generated dataset summary, a summary of the program code, and a randomly selected prompting tip to propose instructions.


Bootstrapped 8 full traces after 13 examples for up to 1 rounds, amounting to 13 attempts.
Error getting source code: unhashable type: 'dict'.

Running without program aware proposer.


2025/08/10 16:27:03 INFO dspy.teleprompt.mipro_optimizer_v2: 
Proposing N=3 instructions...

2025/08/10 16:27:21 INFO dspy.teleprompt.mipro_optimizer_v2: Proposed Instructions for Predictor 0:

2025/08/10 16:27:21 INFO dspy.teleprompt.mipro_optimizer_v2: 0: Given a social media post (title and body) that is already detected to contain stigma toward people with diabetes,
identify the types of stigma present.

2025/08/10 16:27:21 INFO dspy.teleprompt.mipro_optimizer_v2: 1: Identify the types of stigma (experienced, internalized, or perceived) present in a social media post related to diabetes management, and provide a confidence score based on the language used.

2025/08/10 16:27:21 INFO dspy.teleprompt.mipro_optimizer_v2: 2: Identify the types of stigma present in the social media post (title and body) that is already detected to contain stigma toward people with diabetes, as a person who has experienced type 2 diabetes firsthand and understands the emotional toll it can take.

You are 

...

2025/08/10 16:29:41 INFO dspy.evaluate.evaluate: Average Metric: 2.0833333333333335 / 10 (20.8%)
2025/08/10 16:29:41 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 20.83 on minibatch of size 10 with parameters ['Predictor 0: Instruction 2', 'Predictor 0: Few-Shot Set 2'].
2025/08/10 16:29:41 INFO dspy.teleprompt.mipro_optimizer_v2: Minibatch scores so far: []
2025/08/10 16:29:41 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [25.0, 22.5, 20.83, 32.5, 32.5, 20.83]
2025/08/10 16:29:41 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 25.0
2025/08/10 16:29:41 INFO dspy.teleprompt.mipro_optimizer_v2: ========================================


2025/08/10 16:29:41 INFO dspy.teleprompt.mipro_optimizer_v2: == Trial 7 / 8 - Minibatch ==



Average Metric: 2.08 / 10 (20.8%): 100%|██████████| 10/10 [00:00<00:00, 1084.39it/s]

2025/08/10 16:29:41 INFO dspy.evaluate.evaluate: Average Metric: 2.0833333333333335 / 10 (20.8%)
2025/08/10 16:29:41 INFO dspy.teleprompt.mipro_optimizer_v2: Score: 20.83 on minibatch of size 10 with parameters ['Predictor 0: Instruction 2', 'Predictor 0: Few-Shot Set 2'].
2025/08/10 16:29:41 INFO dspy.teleprompt.mipro_optimizer_v2: Minibatch scores so far: []
2025/08/10 16:29:41 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [25.0, 22.5, 20.83, 32.5, 32.5, 20.83, 20.83]
2025/08/10 16:29:41 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 25.0
2025/08/10 16:29:41 INFO dspy.teleprompt.mipro_optimizer_v2: ========================================


2025/08/10 16:29:41 INFO dspy.teleprompt.mipro_optimizer_v2: ===== Trial 8 / 8 - Full Evaluation =====
2025/08/10 16:29:41 INFO dspy.teleprompt.mipro_optimizer_v2: Doing full eval on next top averaging program (Avg Score: 32.5) from minibatch trials...



Average Metric: 3.25 / 10 (32.5%): 100%|██████████| 10/10 [00:00<00:00, 1259.29it/s]

2025/08/10 16:29:41 INFO dspy.evaluate.evaluate: Average Metric: 3.25 / 10 (32.5%)
2025/08/10 16:29:41 INFO dspy.teleprompt.mipro_optimizer_v2: New best full eval score! Score: 32.5
2025/08/10 16:29:42 INFO dspy.teleprompt.mipro_optimizer_v2: Full eval scores so far: [25.0, 22.5, 20.83, 32.5, 32.5, 20.83, 20.83, 32.5]
2025/08/10 16:29:42 INFO dspy.teleprompt.mipro_optimizer_v2: Best full score so far: 32.5
2025/08/10 16:29:42 INFO dspy.teleprompt.mipro_optimizer_v2: =======================
2025/08/10 16:29:42 INFO dspy.teleprompt.mipro_optimizer_v2: 

2025/08/10 16:29:42 INFO dspy.teleprompt.mipro_optimizer_v2: Returning best identified program with score 32.5!


In [46]:
optimized_classify.save('Llama_type_MIPROV2_optimized.json')

In [47]:
del optimized_classify

# COPRO

In [48]:
import litellm
litellm.drop_params = True

from dspy.teleprompt import COPRO

eval_kwargs = dict(num_threads=16, display_progress=True, display_table=0)

copro_teleprompter = COPRO(prompt_model=prompt_gen_lm, metric=jaccard_metric, max_bootstrapped_demos=8, max_labeled_demos=16, breadth=5, depth=3, init_temperature=0, verbose=False)

optimized_classify = copro_teleprompter.compile(type_detector, trainset=t_trainset, eval_kwargs=eval_kwargs)

2025/08/10 16:29:49 INFO dspy.teleprompt.copro_optimizer: Iteration Depth: 1/3.
2025/08/10 16:29:49 INFO dspy.teleprompt.copro_optimizer: At Depth 1/3, Evaluating Prompt Candidate #1/2 for Predictor 1 of 1.






[2025-08-10T16:29:49.464817]

System message:

Your input fields are:
1. `basic_instruction` (str): The initial instructions before optimization
Your output fields are:
1. `proposed_instruction` (str): The improved instructions for the language model
2. `proposed_prefix_for_output_field` (str): The string at the end of the prompt, which will help the model start solving the task
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## basic_instruction ## ]]
{basic_instruction}

[[ ## proposed_instruction ## ]]
{proposed_instruction}

[[ ## proposed_prefix_for_output_field ## ]]
{proposed_prefix_for_output_field}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        You are an instruction optimizer for large language models. I will give you a ``signature`` of fields (inputs and outputs) in English. Your task is to propose an instruction that will lead a good language model to perform the task well. Don't be 

2025/08/10 16:31:12 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


Average Metric: 34.17 / 100 (34.2%): 100%|██████████| 100/100 [02:40<00:00,  1.61s/it]

2025/08/10 16:32:30 INFO dspy.evaluate.evaluate: Average Metric: 34.166666666666664 / 100 (34.2%)
2025/08/10 16:32:30 INFO dspy.teleprompt.copro_optimizer: At Depth 1/3, Evaluating Prompt Candidate #2/2 for Predictor 1 of 1.


...

Average Metric: 20.67 / 57 (36.3%):  57%|█████▋    | 57/100 [01:52<01:05,  1.52s/it]

2025/08/10 16:40:11 WARNING dspy.adapters.json_adapter: Failed to use structured output format, falling back to JSON mode.


Average Metric: 35.08 / 100 (35.1%): 100%|██████████| 100/100 [03:02<00:00,  1.83s/it]

2025/08/10 16:41:20 INFO dspy.evaluate.evaluate: Average Metric: 35.083333333333336 / 100 (35.1%)







[2025-08-10T16:38:17.025516]

System message:

Your input fields are:
1. `attempted_instructions` (str):
Your output fields are:
1. `proposed_instruction` (str): The improved instructions for the language model
2. `proposed_prefix_for_output_field` (str): The string at the end of the prompt, which will help the model start solving the task
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## attempted_instructions ## ]]
{attempted_instructions}

[[ ## proposed_instruction ## ]]
{proposed_instruction}

[[ ## proposed_prefix_for_output_field ## ]]
{proposed_prefix_for_output_field}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        You are an instruction optimizer for large language models. I will give some task instructions I've tried, along with their corresponding validation scores. The instructions are arranged in increasing order based on their scores, where higher scores indicate better quality.


In [49]:
optimized_classify.save('Llama_type_COPRO_optimized.json')

In [50]:
del optimized_classify

# Bootstrap fewshot with optuna

In [51]:
from dspy.teleprompt import BootstrapFewShotWithOptuna

fewshot_optuna_optimizer = BootstrapFewShotWithOptuna(metric=jaccard_metric, max_bootstrapped_demos=8, max_labeled_demos=16, num_candidate_programs=3)

optimized_classify = fewshot_optuna_optimizer.compile(student=type_detector, trainset=t_trainset, valset=t_valset, max_demos = 8)

Going to sample between 1 and 8 traces per predictor.
Will attempt to train 3 candidate sets.


 13%|█▎        | 13/100 [00:00<00:00, 198.11it/s]


Bootstrapped 8 full traces after 13 examples for up to 1 rounds, amounting to 13 attempts.
Average Metric: 2.08 / 10 (20.8%): 100%|██████████| 10/10 [00:19<00:00,  2.00s/it]

2025/08/10 16:41:41 INFO dspy.evaluate.evaluate: Average Metric: 2.0833333333333335 / 10 (20.8%)



Average Metric: 2.25 / 10 (22.5%): 100%|██████████| 10/10 [00:18<00:00,  1.85s/it]

2025/08/10 16:41:59 INFO dspy.evaluate.evaluate: Average Metric: 2.25 / 10 (22.5%)



Average Metric: 2.08 / 10 (20.8%): 100%|██████████| 10/10 [00:19<00:00,  1.94s/it]

2025/08/10 16:42:19 INFO dspy.evaluate.evaluate: Average Metric: 2.0833333333333335 / 10 (20.8%)



Best score: 22.5
Best program: Predict(ClassifyDiabetesStigmaType(title, post -> stigma_type, confidence
    instructions='Given a social media post (title and body) that is already detected to contain stigma toward people with diabetes,\nidentify the types of stigma present.'
    title = Field(annotation=str required=True json_schema_extra={'desc': 'The title of the social media post.', '__dspy_field_type': 'input', 'prefix': 'Title:'})
    post = Field(annotation=str required=True json_schema_extra={'desc': 'The main text or content of the social media post.', '__dspy_field_type': 'input', 'prefix': 'Post:'})
    stigma_type = Field(annotation=List[Literal['experienced stigma', 'perceived stigma', 'anticipated stigma', 'internalized stigma', 'intersectional stigma']] required=True json_schema_extra={'desc': 'List of stigma types present in the post.', '__dspy_field_type': 'output', 'prefix': 'Stigma Type:'})
    confidence = Field(annotation=float required=True json_schema_extra={'d

In [52]:
optimized_classify.save('Llama_type_bootstrap_fewshot_optuna_optimized.json')

In [53]:
del optimized_classify

# Labeled KNN fewshot

In [54]:
from dspy.teleprompt import LabeledFewShot

labeled_fewshot_optimizer = LabeledFewShot(k=8)
optimized_classify = labeled_fewshot_optimizer.compile(student = type_detector, trainset=t_trainset)

In [55]:
optimized_classify.save('Llama_type_labeled_fewshot_optimized.json')

In [56]:
del optimized_classify